In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

customers = [
    (1, "Surya1", "US"),
    (2, "Surya2", "IN"),
    (3, "Surya3", "UK"),
    (4, "Surya4", "US")
]

orders = [
    (101, 1, 500),
    (102, 1, 300),
    (103, 2, 200),
    (104, 5, 400)   # customer_id not present in customers
]

df_customers = spark.createDataFrame(
    customers, ["customer_id", "name", "country"]
)

df_orders = spark.createDataFrame(
    orders, ["order_id", "customer_id", "amount"]
)

df_customers.show()
df_orders.show()


+-----------+------+-------+
|customer_id|  name|country|
+-----------+------+-------+
|          1|Surya1|     US|
|          2|Surya2|     IN|
|          3|Surya3|     UK|
|          4|Surya4|     US|
+-----------+------+-------+

+--------+-----------+------+
|order_id|customer_id|amount|
+--------+-----------+------+
|     101|          1|   500|
|     102|          1|   300|
|     103|          2|   200|
|     104|          5|   400|
+--------+-----------+------+



In [2]:
# Inner join
df_inner = df_customers.join(
    df_orders,
    on="customer_id",
    how="inner"
)

df_inner.show()


+-----------+------+-------+--------+------+
|customer_id|  name|country|order_id|amount|
+-----------+------+-------+--------+------+
|          1|Surya1|     US|     101|   500|
|          1|Surya1|     US|     102|   300|
|          2|Surya2|     IN|     103|   200|
+-----------+------+-------+--------+------+



In [3]:
# left join

df_left = df_customers.join(
    df_orders,
    on="customer_id",
    how="left"
)

df_left.show()


+-----------+------+-------+--------+------+
|customer_id|  name|country|order_id|amount|
+-----------+------+-------+--------+------+
|          1|Surya1|     US|     102|   300|
|          1|Surya1|     US|     101|   500|
|          2|Surya2|     IN|     103|   200|
|          3|Surya3|     UK|    NULL|  NULL|
|          4|Surya4|     US|    NULL|  NULL|
+-----------+------+-------+--------+------+



In [5]:
# Right
df_right = df_customers.join(
    df_orders,
    on="customer_id",
    how="right"
)

df_right.show()


+-----------+------+-------+--------+------+
|customer_id|  name|country|order_id|amount|
+-----------+------+-------+--------+------+
|          1|Surya1|     US|     101|   500|
|          1|Surya1|     US|     102|   300|
|          2|Surya2|     IN|     103|   200|
|          5|  NULL|   NULL|     104|   400|
+-----------+------+-------+--------+------+



In [6]:
# full Outer
df_full = df_customers.join(
    df_orders,
    on="customer_id",
    how="outer"
)

df_full.show()


+-----------+------+-------+--------+------+
|customer_id|  name|country|order_id|amount|
+-----------+------+-------+--------+------+
|          1|Surya1|     US|     101|   500|
|          1|Surya1|     US|     102|   300|
|          2|Surya2|     IN|     103|   200|
|          3|Surya3|     UK|    NULL|  NULL|
|          4|Surya4|     US|    NULL|  NULL|
|          5|  NULL|   NULL|     104|   400|
+-----------+------+-------+--------+------+



In [7]:
# left Semi
# this is used to filter records. Keep rows from the left where a match exists
df_semi = df_customers.join(
    df_orders,
    on="customer_id",
    how="left_semi"
)

df_semi.show()


+-----------+------+-------+
|customer_id|  name|country|
+-----------+------+-------+
|          1|Surya1|     US|
|          2|Surya2|     IN|
+-----------+------+-------+



In [8]:
# left anti join (Find Missing Records)
# Rows in left that have NO match in right
df_anti = df_customers.join(
    df_orders,
    on="customer_id",
    how="left_anti"
)

df_anti.show()


+-----------+------+-------+
|customer_id|  name|country|
+-----------+------+-------+
|          3|Surya3|     UK|
|          4|Surya4|     US|
+-----------+------+-------+



In [10]:
# join on multiple conditions
df_orders2 = df_orders.withColumnRenamed("customer_id", "cust_id")

df_join = df_customers.join(
    df_orders2,
    (df_customers.customer_id == df_orders2.cust_id) & (df_customers.country == "US"),
    "inner"
).drop("cust_id")

df_join.show()


+-----------+------+-------+--------+------+
|customer_id|  name|country|order_id|amount|
+-----------+------+-------+--------+------+
|          1|Surya1|     US|     101|   500|
|          1|Surya1|     US|     102|   300|
+-----------+------+-------+--------+------+



## JOIN with NULL semantics
Null == Null ==> False
but sometimes we want it True. 
For that case we can use <=>

In [20]:
from pyspark.sql.functions import expr
data1 = [
    (1, 1),
    (2, None),
    (3, 2)
]

data2 = [
    (1, 1),
    (None, 3),
    (3, 2)
]

df1 = spark.createDataFrame(
    data1, ["id1", "id2"]
)


df2 = spark.createDataFrame(
    data1, ["id2", "col3"]
)


df1_alias = df1.alias("df1")
df2_alias = df2.alias("df2")

df1_alias.join(df2_alias, expr("df1.id2 <=> df2.id2")).show()


+---+---+---+----+
|id1|id2|id2|col3|
+---+---+---+----+
|  1|  1|  1|   1|
|  3|  2|  2|NULL|
+---+---+---+----+



In [23]:
joined_df = df1.join(df2, df1["id2"].eqNullSafe(df2["id2"]))
joined_df.show()

+---+---+---+----+
|id1|id2|id2|col3|
+---+---+---+----+
|  1|  1|  1|   1|
|  3|  2|  2|NULL|
+---+---+---+----+



# SKEWED JOIN Handling
Problem:

    * One key has millions of rows → stragglers

Solutions:

    * Broadcast small table
    * Salt the key
    * AQE (Adaptive Query Execution)

# Broadcast

In [12]:
from pyspark.sql.functions import broadcast

df_join = df_orders.join(
    broadcast(df_customers),
    "customer_id",
    "inner"
)

df_join.show()

+-----------+--------+------+------+-------+
|customer_id|order_id|amount|  name|country|
+-----------+--------+------+------+-------+
|          1|     101|   500|Surya1|     US|
|          1|     102|   300|Surya1|     US|
|          2|     103|   200|Surya2|     IN|
+-----------+--------+------+------+-------+



## RANGE / TIME WINDOW JOIN
Use case

Events within time window

Slowly changing dimensions

In [27]:
from pyspark.sql.functions import to_timestamp
events_data = [
    (1, "2025-01-05 10:00:00"),
    (1, "2025-01-15 12:00:00"),
    (2, "2025-01-20 09:00:00"),
    (3, "2025-01-25 14:00:00")
]

df_events = spark.createDataFrame(
    events_data,
    ["id", "ts"]
).withColumn("ts", to_timestamp("ts"))

df_events.show(truncate=False)



+---+-------------------+
|id |ts                 |
+---+-------------------+
|1  |2025-01-05 10:00:00|
|1  |2025-01-15 12:00:00|
|2  |2025-01-20 09:00:00|
|3  |2025-01-25 14:00:00|
+---+-------------------+



In [28]:
dim_data = [
    (1, "A", "2025-01-01", "2025-01-10"),
    (1, "B", "2025-01-11", "2025-01-31"),
    (2, "X", "2025-01-01", "2025-01-31")
]

df_dim = spark.createDataFrame(
    dim_data,
    ["id", "category", "start", "end"]
).withColumn("start", to_timestamp("start")) \
 .withColumn("end", to_timestamp("end"))

df_dim.show(truncate=False)

+---+--------+-------------------+-------------------+
|id |category|start              |end                |
+---+--------+-------------------+-------------------+
|1  |A       |2025-01-01 00:00:00|2025-01-10 00:00:00|
|1  |B       |2025-01-11 00:00:00|2025-01-31 00:00:00|
|2  |X       |2025-01-01 00:00:00|2025-01-31 00:00:00|
+---+--------+-------------------+-------------------+



In [30]:

df_events.join(
    df_dim,
    (df_events.id == df_dim.id) &
    (df_events.ts.between(df_dim.start, df_dim.end))
).show()

+---+-------------------+---+--------+-------------------+-------------------+
| id|                 ts| id|category|              start|                end|
+---+-------------------+---+--------+-------------------+-------------------+
|  1|2025-01-05 10:00:00|  1|       A|2025-01-01 00:00:00|2025-01-10 00:00:00|
|  1|2025-01-15 12:00:00|  1|       B|2025-01-11 00:00:00|2025-01-31 00:00:00|
|  2|2025-01-20 09:00:00|  2|       X|2025-01-01 00:00:00|2025-01-31 00:00:00|
+---+-------------------+---+--------+-------------------+-------------------+

